# RAG Evaluation — AI Engineering Interview Prep

RAGAS-style metrics: faithfulness, answer relevancy, context recall, context precision.

**Install**: `pip install anthropic sentence-transformers`

In [ ]:
import anthropic
import os
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-4-5-20251001"
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def llm(prompt, max_tokens=400):
    resp = client.messages.create(
        model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.content[0].text

# Sample RAG outputs to evaluate
test_samples = [
    {
        "question": "What is the formula for scaled dot-product attention?",
        "contexts": [
            "Transformers use scaled dot-product attention: Attention(Q,K,V) = softmax(QK^T/sqrt(d_k))V.",
            "The scaling factor sqrt(d_k) prevents the dot product from growing too large for large dimensions."
        ],
        "answer": "Attention(Q,K,V) = softmax(QK^T/sqrt(d_k))V, where d_k is the dimension of the key vectors.",
        "ground_truth": "The formula is Attention(Q,K,V) = softmax(QK^T/sqrt(d_k))V."
    },
    {
        "question": "How does LoRA reduce training parameters?",
        "contexts": [
            "LoRA injects trainable low-rank matrices into frozen pretrained weights: W + BA.",
            "QLoRA combines 4-bit quantization with LoRA to fit 70B models on a single GPU."
        ],
        "answer": "LoRA uses low-rank decomposition: instead of training the full weight matrix W, it trains two small matrices B and A where the update is BA. This reduces parameters from d*k to r*(d+k) where r << min(d,k).",
        "ground_truth": "LoRA freezes pretrained weights and adds trainable low-rank decomposition matrices B and A, so only r*(d+k) parameters are trained instead of d*k."
    },
    {
        "question": "What optimizer is best for transformers?",
        "contexts": [
            "Batch normalization normalizes layer inputs to have zero mean and unit variance.",
            "Dropout regularization randomly sets neuron activations to zero during training."
        ],
        "answer": "Adam optimizer with warmup is the standard choice for transformer training.",
        "ground_truth": "AdamW with a linear warmup schedule is most commonly used for training transformers."
    },
]

print(f"Evaluating {len(test_samples)} RAG samples")

## 1. Faithfulness — Is the answer grounded in context?

In [ ]:
def measure_faithfulness(answer, contexts):
    """Check what fraction of answer claims are supported by contexts."""
    context_str = "\n".join(f"- {c}" for c in contexts)

    prompt = f"""Break the answer into atomic claims. For each claim, check if it is supported by the context.
Output ONLY valid JSON: {{"claims": [{{"claim": "...", "supported": true/false}}], "faithfulness_score": 0.0-1.0}}

Context:
{context_str}

Answer: {answer}"""

    raw = llm(prompt)
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(clean)
    except:
        return {"faithfulness_score": 0.5, "error": raw[:100]}

print("Faithfulness scores:")
for sample in test_samples:
    result = measure_faithfulness(sample["answer"], sample["contexts"])
    score = result.get("faithfulness_score", "?")
    print(f"  Q: {sample['question'][:60]}")
    print(f"     Faithfulness: {score}")

## 2. Answer Relevancy — Does the answer address the question?

In [ ]:
def measure_answer_relevancy(question, answer, n=3):
    """Generate N questions from the answer, measure semantic similarity to original question."""
    prompt = f"""Given this answer, generate {n} questions that this answer would address.
Output ONLY the questions, one per line.

Answer: {answer}"""

    raw = llm(prompt, max_tokens=200)
    generated_qs = [q.strip().lstrip('0123456789.-) ') for q in raw.strip().split('\n') if q.strip()][:n]

    # Embed original question and generated questions
    all_qs = [question] + generated_qs
    embeddings = embed_model.encode(all_qs, normalize_embeddings=True)
    sims = cosine_similarity([embeddings[0]], embeddings[1:])[0]

    return float(np.mean(sims)), generated_qs

print("Answer Relevancy scores:")
for sample in test_samples:
    score, gen_qs = measure_answer_relevancy(sample["question"], sample["answer"])
    print(f"  Q: {sample['question'][:60]}")
    print(f"     Relevancy: {score:.4f}")

## 3. Context Recall — Does context contain ground truth info?

In [ ]:
def measure_context_recall(ground_truth, contexts):
    """Check what fraction of ground truth claims are in contexts."""
    context_str = "\n".join(f"- {c}" for c in contexts)

    prompt = f"""Break the ground truth into atomic claims. For each claim, check if it is present in the context.
Output ONLY valid JSON: {{"claims": [{{"claim": "...", "in_context": true/false}}], "recall_score": 0.0-1.0}}

Context:
{context_str}

Ground truth: {ground_truth}"""

    raw = llm(prompt)
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(clean)
    except:
        return {"recall_score": 0.5}

print("Context Recall scores:")
for sample in test_samples:
    result = measure_context_recall(sample["ground_truth"], sample["contexts"])
    score = result.get("recall_score", "?")
    print(f"  Q: {sample['question'][:60]}")
    print(f"     Context Recall: {score}")

## 4. Full Evaluation Summary

In [ ]:
# RAGAS metric overview (without calling APIs for all combos)
ragas_metrics = {
    "Faithfulness": {
        "inputs": "answer + contexts",
        "measures": "Are answer claims supported by context?",
        "range": "0-1, higher is better",
        "detects": "Hallucination"
    },
    "Answer Relevancy": {
        "inputs": "question + answer",
        "measures": "Does the answer address the question?",
        "range": "0-1, higher is better",
        "detects": "Off-topic or incomplete answers"
    },
    "Context Recall": {
        "inputs": "ground_truth + contexts",
        "measures": "Fraction of GT claims present in context",
        "range": "0-1, higher is better",
        "detects": "Retrieval gaps"
    },
    "Context Precision": {
        "inputs": "question + ground_truth + contexts",
        "measures": "Fraction of retrieved context that is useful",
        "range": "0-1, higher is better",
        "detects": "Noisy/irrelevant retrieved docs"
    },
}

print(f"{'Metric':<25} {'Inputs':<30} {'Detects'}")
print("-" * 80)
for metric, info in ragas_metrics.items():
    print(f"{metric:<25} {info['inputs']:<30} {info['detects']}")

## Interview Tips

- **Faithfulness vs Relevancy**: faithfulness checks grounding; relevancy checks if answer is on-topic.
- **Context Recall vs Precision**: recall=coverage (did you retrieve the right docs?); precision=signal-to-noise.
- **Reference-free**: faithfulness and answer relevancy don't need ground truth — great for production.
- **Reference-required**: context recall and precision need ground truth — use for offline evaluation.
- **RAGAS library**: `ragas.evaluate()` wraps all 4 metrics. Uses gpt-4 or claude as the judge.
- **Tradeoffs**: high faithfulness + low relevancy = grounded but off-topic. High relevancy + low faithfulness = on-topic but hallucinated.